### 00 — Data generation (setup, outside the medallion pipeline)

Simulates SNCF Gares & Connexions station footfall data. This is a seed step to make the project fully reproducible.

**Output:** a single CSV in the landing zone
`/Volumes/sncf_gc/bronze/landing/frequentation.csv`, then ingested by `01_bronze`.


In [0]:
import numpy as np
import pandas as pd
from datetime import date, timedelta

SEED = 42
np.random.seed(SEED)  # reproductibilité

LANDING_PATH = "/Volumes/sncf_gc/bronze/landing/frequentation.csv"
ANNEE_DEBUT, ANNEE_FIN = date(2023, 1, 1), date(2023, 12, 31)
TRANCHES = [8, 18]                       # pic du matin, pic du soir
BASE_SEGMENT = {"A": 5000, "B": 1800, "C": 500}  # affluence type par catégorie de gare

## 1. Référentiel des gares
 City names
are chosen to match the INSEE reference during the Silver join.

In [0]:
GARES = [
    # gare_id, nom_gare, ville, region, type_gare, segment, lat, lon
    (1,  "Paris Gare de Lyon",     "Paris",      "Île-de-France",              "Terminus", "A", 48.844,  2.373),
    (2,  "Paris Montparnasse",     "Paris",      "Île-de-France",              "Terminus", "A", 48.840,  2.319),
    (3,  "Marseille Saint-Charles","Marseille",  "Provence-Alpes-Côte d'Azur", "Terminus", "A", 43.302,  5.380),
    (4,  "Lyon Part-Dieu",         "Lyon",       "Auvergne-Rhône-Alpes",       "Passage",  "A", 45.760,  4.859),
    (5,  "Toulouse Matabiau",      "Toulouse",   "Occitanie",                  "Passage",  "B", 43.611,  1.453),
    (6,  "Bordeaux Saint-Jean",    "Bordeaux",   "Nouvelle-Aquitaine",         "Passage",  "B", 44.826, -0.556),
    (7,  "Lille Flandres",         "Lille",      "Hauts-de-France",            "Terminus", "A", 50.637,  3.071),
    (8,  "Strasbourg",             "Strasbourg", "Grand Est",                  "Passage",  "B", 48.585,  7.735),
    (9,  "Nantes",                 "Nantes",     "Pays de la Loire",           "Terminus", "B", 47.217, -1.542),
    (10, "Rennes",                 "Rennes",     "Bretagne",                   "Passage",  "B", 48.103, -1.672),
    (11, "Nice-Ville",             "Nice",       "Provence-Alpes-Côte d'Azur", "Passage",  "B", 43.704,  7.262),
    (12, "Montpellier Saint-Roch", "Montpellier","Occitanie",                  "Passage",  "B", 43.605,  3.881),
    (13, "Le Havre",               "Le Havre",   "Normandie",                  "Terminus", "C", 49.493,  0.123),
    (14, "Dijon-Ville",            "Dijon",      "Bourgogne-Franche-Comté",    "Passage",  "C", 47.323,  5.027),
    (15, "Reims",                  "Reims",      "Grand Est",                  "Passage",  "C", 49.259,  4.025),
    (16, "Tours",                  "Tours",      "Centre-Val de Loire",        "Jonction", "C", 47.390,  0.694),
    (17, "Grenoble",               "Grenoble",   "Auvergne-Rhône-Alpes",       "Passage",  "C", 45.191,  5.717),
    (18, "Angers Saint-Laud",      "Angers",     "Pays de la Loire",           "Passage",  "C", 47.464, -0.557),
]
ref = pd.DataFrame(GARES, columns=["gare_id","nom_gare","ville","region",
                                   "type_gare","segment","latitude","longitude"])
# code UIC fictif mais cohérent
ref["code_uic"] = "87" + (700000 + ref["gare_id"]).astype(str).str[-6:]

## 2. Footfall events
One row per (station × day × time slot). Volume scales with station tier (A > B > C),
peaks at rush hours, and drops on weekends .

In [0]:
def generer_evenements(ref):
    """Génère les événements de fréquentation horaire pour chaque gare.

    L'affluence suit une loi normale autour d'une base (catégorie de gare),
    pondérée par un facteur week-end et un facteur de pic horaire.

    Args:
        ref: DataFrame du référentiel des gares.
    Returns:
        DataFrame propre [colonnes métier], sans anomalie.
    """
    jours = [ANNEE_DEBUT + timedelta(d) for d in range((ANNEE_FIN - ANNEE_DEBUT).days + 1)]
    lignes = []
    for _, g in ref.iterrows():
        base = BASE_SEGMENT[g["segment"]]
        for j in jours:
            f_we = 0.55 if j.weekday() >= 5 else 1.0     # week-end plus calme
            for t in TRANCHES:
                f_pic = 1.0 if t == 8 else 0.85
                nv  = int(max(0, np.random.normal(base * f_we * f_pic, base * 0.12)))
                nnv = int(max(0, np.random.normal(nv * 0.15, nv * 0.05)))
                lignes.append([g["gare_id"], g["code_uic"], g["nom_gare"], g["ville"],
                               g["region"], g["type_gare"], g["segment"],
                               g["latitude"], g["longitude"],
                               j.isoformat(), t, nv, nnv])
    cols = ["gare_id","code_uic","nom_gare","ville","region","type_gare","segment",
            "latitude","longitude","date","heure_tranche","nb_voyageurs","nb_non_voyageurs"]
    return pd.DataFrame(lignes, columns=cols)


df = generer_evenements(ref)
print(f"{len(df)} lignes propres générées")

## 3. Injected anomalies
A small fraction of rows is deliberately degraded so the pipeline has something to
clean: duplicates, mixed/invalid date formats, inconsistent casing, negative values,
invalid categories, and missing values. Technical issues are handled in Bronze,
business-rule issues in Silver.

In [0]:
def injecter_anomalies(df):
    """Dégrade une fraction des lignes pour exercer le pipeline de nettoyage.

    Les anomalies sont calibrées pour rester minoritaires (réalistes) et couvrir
    chaque étape de Bronze et Silver.
    """
    df = df.astype({"date": "string"})
    def echantillon(frac):
        return np.random.choice(df.index, int(len(df) * frac), replace=False)

    # dates au format dd/MM/yyyy (10 %)
    for i in echantillon(0.10):
        a, m, j = df.at[i, "date"].split("-")
        df.at[i, "date"] = f"{j}/{m}/{a}"
    # dates invalides (0,5 %)
    for i in echantillon(0.005):
        df.at[i, "date"] = "32/13/2023"
    # valeurs négatives (2 %)
    for i in echantillon(0.02):
        df.at[i, "nb_voyageurs"] = -abs(int(df.at[i, "nb_voyageurs"]))
    # valeurs manquantes
    df.loc[echantillon(0.03), "nb_voyageurs"] = np.nan
    df.loc[echantillon(0.03), "nb_non_voyageurs"] = np.nan
    df.loc[echantillon(0.01), "latitude"] = np.nan
    # catégories invalides (1 % chacune)
    sel = echantillon(0.01)
    df.loc[sel, "segment"] = np.random.choice(["D", "Z", "X"], size=len(sel))
    df.loc[echantillon(0.01), "type_gare"] = "Inconnu"
    # casse incohérente (10 %)
    for c in ["region", "ville", "type_gare"]:
        sel = echantillon(0.10)
        df.loc[sel, c] = df.loc[sel, c].str.lower()
    # doublons exacts (1 %)
    df = pd.concat([df, df.loc[echantillon(0.01)].copy()], ignore_index=True)

    # entiers nullables → CSV propre (pas de "3048.0", null = vide)
    for c in ["gare_id", "heure_tranche", "nb_voyageurs", "nb_non_voyageurs"]:
        df[c] = pd.to_numeric(df[c], errors="coerce").astype("Int64")
    return df


df = injecter_anomalies(df)
print(f"{len(df)} lignes après injection d'anomalies")

## 4. Write to the landing zone


In [0]:
spark.sql("CREATE VOLUME IF NOT EXISTS sncf_gc.bronze.landing")
df.to_csv(LANDING_PATH, index=False)
print(f"✅ écrit : {LANDING_PATH}")

In [0]:
relu = spark.read.option("header", True).csv(LANDING_PATH)
print(f"Relu depuis la landing : {relu.count()} lignes")
display(relu.limit(5))